In [ ]:
# Cell 1: Configure dataset root and define reusable Kaggle download helpers
from pathlib import Path
import os
import shutil
import sys
import subprocess

DATASET_ROOT = Path('/scratch/rameyjm7/datasets').resolve()
DATASET_ROOT.mkdir(parents=True, exist_ok=True)

print(f'Dataset root: {DATASET_ROOT}')

try:
    import kagglehub
except ImportError:
    print('kagglehub not found; installing...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'kagglehub'])
    import kagglehub


def download_and_copy(dataset_ref: str, target_dir: Path, allowed_suffixes: tuple[str, ...]):
    target_dir.mkdir(parents=True, exist_ok=True)
    download_path = Path(kagglehub.dataset_download(dataset_ref))
    print(f'Downloaded {dataset_ref} to cache: {download_path}')

    copied = 0
    for src in download_path.rglob('*'):
        if src.is_file() and src.suffix.lower() in allowed_suffixes:
            dst = target_dir / src.name
            if dst.exists() and dst.stat().st_size == src.stat().st_size:
                continue
            shutil.copy2(src, dst)
            copied += 1
            print(f'Copied: {src} -> {dst}')

    if copied == 0:
        print(f'No new files copied for {dataset_ref}.')
    else:
        print(f'Copied {copied} files for {dataset_ref} into {target_dir}.')


In [ ]:
# Cell 2: Download RML2018 artifacts into /scratch/rameyjm7/datasets/RML2018
rml2018_dir = DATASET_ROOT / 'RML2018'
download_and_copy(
    dataset_ref='pinxau1000/radioml2018',
    target_dir=rml2018_dir,
    allowed_suffixes=('.h5', '.hdf5', '.txt', '.pkl'),
)


In [ ]:
# Cell 3: Download RML2016 artifacts into /scratch/rameyjm7/datasets/RML2016
rml2016_dir = DATASET_ROOT / 'RML2016'
download_and_copy(
    dataset_ref='zaslee/rml2016-10a',
    target_dir=rml2016_dir,
    allowed_suffixes=('.pkl', '.bz2', '.tar', '.gz'),
)


In [ ]:
# Cell 4: Download DeepRadar2022 artifacts into /scratch/rameyjm7/datasets/DeepRadar2022
deepradar_dir = DATASET_ROOT / 'DeepRadar2022'
download_and_copy(
    dataset_ref='khilian/deepradar',
    target_dir=deepradar_dir,
    allowed_suffixes=('.mat',),
)


In [ ]:
# Cell 5: Download Noisy Drone RF Signal Classification v2 artifacts into /scratch/rameyjm7/datasets/NoisyDroneRFv2
noisy_drone_rf_dir = DATASET_ROOT / 'NoisyDroneRFv2'
download_and_copy(
    dataset_ref='sgluege/noisy-drone-rf-signal-classification-v2',
    target_dir=noisy_drone_rf_dir,
    allowed_suffixes=('.pt', '.csv'),
)


In [ ]:
# Cell 6: Report expected files and print symlink command for repo data/ if needed
expected = {
    'RML2016': ['RML2016.10a_dict.pkl'],
    'RML2018': ['GOLD_XYZ_OSC.0001_1024.hdf5', 'classes.txt', 'classes-fixed.txt'],
    'DeepRadar2022': ['X_test.mat', 'Y_test.mat', 'lbl_test.mat'],
    'NoisyDroneRFv2': ['class_stats.csv', 'SNR_stats.csv'],
}

for ds_name, names in expected.items():
    ds_dir = DATASET_ROOT / ds_name
    print(f'
{ds_name}: {ds_dir}')
    for name in names:
        p = ds_dir / name
        print(f'  {name}:', 'FOUND' if p.exists() else 'MISSING')
    if ds_name == 'NoisyDroneRFv2':
        pt_count = len(list(ds_dir.rglob('IQdata_sample*_target*_snr*.pt'))) if ds_dir.exists() else 0
        print(f'  IQdata_sample*_target*_snr*.pt files: {pt_count}')

print('
If repo data symlink needs reset:')
print('cd /home/rameyjm7/workspace/ML-wireless-signal-classification')
print('rm -f data && ln -s /scratch/rameyjm7/datasets data')


In [ ]:
# Cell 7: Write local data path config for notebooks 20/30s/40 to consume
import yaml

repo_root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
config_path = repo_root / 'configs' / 'local_data_paths.yaml'
config_path.parent.mkdir(parents=True, exist_ok=True)

local_paths = {
    'version': 1,
    'dataset_root': str(DATASET_ROOT),
    'datasets': {
        'rml2016': {
            'pkl': str(DATASET_ROOT / 'RML2016' / 'RML2016.10a_dict.pkl'),
        },
        'rml2018': {
            'hdf5': str(DATASET_ROOT / 'RML2018' / 'GOLD_XYZ_OSC.0001_1024.hdf5'),
            'classes': str(DATASET_ROOT / 'RML2018' / 'classes.txt'),
            'classes_fixed': str(DATASET_ROOT / 'RML2018' / 'classes-fixed.txt'),
        },
        'deepradar2022': {
            'x_test': str(DATASET_ROOT / 'DeepRadar2022' / 'X_test.mat'),
            'y_test': str(DATASET_ROOT / 'DeepRadar2022' / 'Y_test.mat'),
            'lbl_test': str(DATASET_ROOT / 'DeepRadar2022' / 'lbl_test.mat'),
            'x_train': str(DATASET_ROOT / 'DeepRadar2022' / 'X_train.mat'),
            'y_train': str(DATASET_ROOT / 'DeepRadar2022' / 'Y_train.mat'),
            'lbl_train': str(DATASET_ROOT / 'DeepRadar2022' / 'lbl_train.mat'),
            'x_val': str(DATASET_ROOT / 'DeepRadar2022' / 'X_val.mat'),
            'y_val': str(DATASET_ROOT / 'DeepRadar2022' / 'Y_val.mat'),
            'lbl_val': str(DATASET_ROOT / 'DeepRadar2022' / 'lbl_val.mat'),
        },
        'noisy_drone_rf_v2': {
            'data_dir': str(DATASET_ROOT / 'NoisyDroneRFv2'),
            'class_stats': str(DATASET_ROOT / 'NoisyDroneRFv2' / 'class_stats.csv'),
            'snr_stats': str(DATASET_ROOT / 'NoisyDroneRFv2' / 'SNR_stats.csv'),
        },
    },
}

with config_path.open('w', encoding='utf-8') as f:
    yaml.safe_dump(local_paths, f, sort_keys=False)

print(f'Wrote local data config: {config_path}')
print('You can now run notebooks 20/30s/40 without hardcoded repo/data paths.')
